# Built with Meta Llama 3

In [ ]:
!pip install --upgrade transformers bitsandbytes accelerate

In [ ]:
import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import default_data_collator
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

import torch
from datasets import Dataset
from datasets import load_dataset

import gc
import time
import re
import io

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
all_qa = load_dataset("cardiffnlp/databench", name="qa", split="train")

# **PAL**

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
def load_ds(i):
  ds_id = all_qa['dataset'][i]
  df = pd.read_parquet(
      f'hf://datasets/cardiffnlp/databench/data/{ds_id}/sample.parquet'
  )
  col = df.columns

  df_str = df.to_csv(sep='|', index=False)

  qa = pd.read_parquet(
      f'hf://datasets/cardiffnlp/databench/data/{ds_id}/qa.parquet'
  )
  questions = qa['question'].tolist()
  qa_answers = qa['sample_answer'].tolist()

  return df, col, df_str, qa_answers, questions

In [ ]:
for i in [0, 130, 240, 360, 700]:
  df, col, data, answers, questions = load_ds(i)

  if isinstance(df, str):
    df = pd.read_csv(io.StringIO(df), sep='|')

  schema_context = f'Data Columns: {col}'
  temp1 = []
  temp2 = []

  match i:
    case 0:
      for l in [0, 1, 2, 6, 13, 17]:
        temp1.append(questions[l])
        temp2.append(answers[l])
    case 130:
      for l in [0, 1, 2, 5, 13, 14]:
        temp1.append(questions[l])
        temp2.append(answers[l])
    case 240:
      for l in [0, 3, 7, 8, 13, 16]:
        temp1.append(questions[l])
        temp2.append(answers[l])
    case 360:
      for l in [2, 6, 8, 11, 14, 18]:
        temp1.append(questions[l])
        temp2.append(answers[l])
    case 700:
      for l in [0, 1, 3, 10, 14, 20]:
        temp1.append(questions[l])
        temp2.append(answers[l])

  questions = temp1
  answers = temp2

  for q_idx in range(len(questions)):

    current_data_context = data

    prompt = f"""My dataframe is called df and it possesses the following columns {col}.
            Using only pandas and numpy libraries give me code to answer the following question: {questions[q_idx]}
            Don't give me any text whatsoever aside from code.
            Output ONLY executable code wrapped in triple dashes (---)."""
    answer = generator(
        prompt, temperature=0.1, max_new_tokens=256, return_full_text=False)[0]['generated_text'].strip()
    print(f'Prompt: {prompt}')
    print(f'AI Proposed Code: {answer}')

    if '---' in answer:
      parts = answer.split('---')
      answer = parts[1]
      answer = re.sub(r'^(python|Python)\s*', '', answer).strip()

    try:
      if '---' in answer:
        parts = answer.split('---')
        answer = parts[1] if len(parts) > 1 else parts[0]
      answer = re.sub(r'^(python|Python)\s*', '', answer).strip()

      code_lines = [
          line
          for line in answer.split('\n')
          if not line.strip().startswith(('import ', 'from '))
      ]
      clean_code = '\n'.join(code_lines).strip()

      env = {'df': df, 'pd': pd, 'np': np}

      if '=' not in clean_code and '\n' not in clean_code:
        observation = eval(clean_code, env)
      else:
        exec(clean_code, env)
        if 'result' in env:
          observation = env['result']
        else:
          assigned_vars = [
              k for k in env.keys() if k not in ['df', 'pd', 'np', '__builtins__']
          ]
          if assigned_vars:
            observation = env[assigned_vars[-1]]
          else:
            last_line = [l for l in code_lines if l.strip()][-1]
            observation = eval(last_line, env)

      print(f'Execution Result: {observation}')

    except Exception as e:
      observation = f'Error: {str(e)}'
      print(observation)

    print(f'Actual Answer: {answers[q_idx]}')
    print('--------------------------------------------------------------')

# **ReAct**

In [ ]:
def load_ds(i):
  ds_id = all_qa['dataset'][i]
  df = pd.read_parquet(
      f'hf://datasets/cardiffnlp/databench/data/{ds_id}/sample.parquet'
  )
  col = df.columns

  df_str = df.to_csv(sep='|', index=False)

  qa = pd.read_parquet(
      f'hf://datasets/cardiffnlp/databench/data/{ds_id}/qa.parquet'
  )
  questions = qa['question'].tolist()
  qa_answers = qa['sample_answer'].tolist()

  return df, col, df_str, qa_answers, questions

def execute_safely(raw_code, df_in):
    code = raw_code.strip()

    if "```" in code:
        match = re.search(r"```(?:python)?\s*(.*?)\s*```", code, re.DOTALL)
        if match:
            code = match.group(1)
    elif "---" in code:
        parts = code.split("---")
        code = parts[1] if len(parts) > 1 else parts[0]

    code = re.sub(r'^(python|Python)\s*', '', code).strip()

    lines = [
        l for l in code.split('\n')
        if not l.strip().startswith(('import ', 'from ', 'df =', 'df='))
        and not l.strip().startswith('#')
    ]
    clean_code = '\n'.join(lines).strip()

    env = {'df': df_in.copy(), 'pd': pd, 'np': np}

    try:
        if '=' not in clean_code and '\n' not in clean_code:
            return eval(clean_code, env)

        exec(clean_code, env)

        if 'result' in env:
            res = env['result']
            return res(env['df']) if callable(res) else res

        assigned = [k for k in env if k not in ['df', 'pd', 'np', '__builtins__']]
        if assigned:
            return env[assigned[-1]]

        non_empty = [l for l in lines if l.strip()]
        if non_empty and '=' not in non_empty[-1]:
            return eval(non_empty[-1], env)

        return 'Executed successfully (no explicit return)'
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
pad_id = generator.tokenizer.pad_token_id or generator.tokenizer.eos_token_id

for i in [0, 130, 240, 360, 700]:
    df, col, data, answers, questions = load_ds(i)

    if isinstance(df, str):
        df = pd.read_csv(io.StringIO(df), sep='|')

    schema_context = f'Data Columns: {col}'
    temp1 = []
    temp2 = []

    match i:
        case 0:
            for l in [0, 1, 2, 6, 13, 17]:
                temp1.append(questions[l])
                temp2.append(answers[l])
        case 130:
            for l in [0, 1, 2, 5, 13, 14]:
                temp1.append(questions[l])
                temp2.append(answers[l])
        case 240:
            for l in [0, 3, 7, 8, 13, 16]:
                temp1.append(questions[l])
                temp2.append(answers[l])
        case 360:
            for l in [2, 6, 8, 11, 14, 18]:
                temp1.append(questions[l])
                temp2.append(answers[l])
        case 700:
            for l in [0, 1, 3, 10, 14, 20]:
                temp1.append(questions[l])
                temp2.append(answers[l])

    questions = temp1
    answers = temp2

    for q_idx in range(len(questions)):
        if q_idx >= len(questions):
            continue

        q = questions[q_idx]
        actual_ans = answers[q_idx]

        thought_prompt = f"""My dataframe is called df and it possesses the following columns {list(col)}.\n
                          Task: {q}\n
                          Do NOT write code, just break down the step by step filtering process for how you would achieve this."""

        thought_answer = generator(
            thought_prompt,
            temperature=0.1,
            max_new_tokens=512,
            return_full_text=False,
            pad_token_id=pad_id,
            do_sample=False
        )[0]['generated_text'].strip()

        skip_first = 1
        observation_answer = "FAILURE"
        action_answer = ""
        current_obs = ""
        max_retries = 3

        while "FAILURE" in observation_answer and max_retries > 0:
            max_retries -= 1

            action_prompt = f"""My dataframe is called df and it possesses the following columns {list(col)}.\n
                            Task: {q}\n
                            Follow the steps of the Thought to craft code. Assign your final computed answer to a variable called `result`.
                            Do not output anything other than pure python code wrapped in ```python ... ```.
                            Thought: {thought_answer}\n"""

            if "FAILURE" in observation_answer and skip_first == 0:
                action_prompt += f"""Your last attempt contained a mistake. Review the feedback and fix the code:
                                  Correction Feedback: {observation_answer}"""

            action_answer = generator(
                action_prompt,
                temperature=0.1,
                max_new_tokens=512,
                return_full_text=False,
                pad_token_id=pad_id,
                do_sample=False
            )[0]['generated_text'].strip()
            skip_first = 0

            current_obs = execute_safely(action_answer, df)

            observation_prompt = f"""My dataframe is called df with columns {list(col)}.\n
                                    Task: {q}\n
                                    Code: {action_answer}\n
                                    Execution Output: {current_obs}\n
                                    Examine if the code executes cleanly and achieves the goal without errors.
                                    If it succeeded and gave the correct answer, respond strictly with 'SUCCESS'.
                                    If there is a runtime error or logic error, respond 'FAILURE' followed by a short explanation."""

            if str(current_obs).startswith("Error:"):
                observation_answer = f"FAILURE: Code crashed with {current_obs}"
            else:
                observation_answer = generator(
                    observation_prompt,
                    temperature=0.1,
                    max_new_tokens=256,
                    return_full_text=False,
                    pad_token_id=pad_id,
                    do_sample=False
                )[0]['generated_text'].strip()

        print(f"Dataset {i} | Q: {q}")
        print(f"Proposed Code:\n{action_answer}")
        print(f"Verifier Verdict: {'SUCCESS' if 'SUCCESS' in observation_answer else 'EXHAUSTED RETRIES'}")
        print(f"Execution Result: {current_obs}")
        print(f"Actual Answer:    {actual_ans}")
        print("-" * 60)

# **Tri-Agent**

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
def generate_safe(prompt: str, max_new_tokens: int = 512) -> str:
  messages = [{"role": "user", "content": prompt}]
  inputs = tokenizer.apply_chat_template(
      messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
  ).to(model.device)

  prompt_length = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

  return tokenizer.decode(
      outputs[0][prompt_length:], skip_special_tokens=True
  ).strip()


def load_ds(i):
    ds_id = all_qa["dataset"][i]
    df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/sample.parquet")

    df.columns = [re.sub(r'<gx:[^>]+>', '', str(c)).strip() for c in df.columns]
    col = list(df.columns)

    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype(str).str.slice(0, 100)

    df_str = df.to_csv(sep="|", index=False)

    qa_df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/qa.parquet")
    answers = qa_df["sample_answer"].tolist()
    questions = qa_df["question"].tolist()

    return df, col, df_str, answers, questions

In [ ]:
class Thinker:
    role = """You are a data scientist named Thinker. Read the Dataset Structure and explain concisely:
1. Which column(s) to inspect.
2. The exact value or aggregate operation required to complete the target task.
Do NOT write code. Keep explanations under 3 sentences."""

    def generate(self, step, schema_context):
        prompt = f"### ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"
        return generate_safe(prompt, 256)


class Performer:
    role = """You are a seasoned data scientist and python programmer named Performer.
      Your task is to use the Relevant Terms to produce pandas code that will realize the request of the TARGET TASK.
  CRITICAL CONSTRAINT RULES:
  1. Do NOT initialize, mock, copy, or create any new dataframes or dictionaries (e.g., NEVER write statements like `df = pd.DataFrame(...)`, `sample_df = ...`, or mock data arrays).
  2. If filtering for a subset or a single row, KEEP IT AS A DATAFRAME (e.g., use `df[df['col'] == val].head(1)` instead of `.iloc[0]` or `.loc[...]`) so subsequent chained steps can continue filtering it.
  3. You may use the original df or variables defined in the PREVIOUS WORKING CODE. Save the latest step's final output (DataFrame, Series, boolean, or scalar value) to result_df.
  4. Always add `na=False` inside your `.str.contains()` operations to avoid execution errors on empty data rows.
  5. Do NOT use pd.read_csv() or load any files.
  6. Wrap your code inside ```python and ``` blocks."""

    def generate(self, schema_context, step, answer_thinker, feedback_note, previous_code):
        prompt = f"### ROLE:\n{self.role}\n\n### SCHEMA:\n{schema_context}\n\n### TARGET TASK:\n{step}\n\n### METHOD:\n{answer_thinker}"

        if previous_code:
            prompt += f"\n\n### PREVIOUS WORKING CODE:\n```python\n{previous_code}\n```\nBuild directly on top of variables from the previous code."

        if feedback_note:
            prompt += f"\n\n### RUNTIME ERROR FEEDBACK:\n{feedback_note}\nFix the code based on this error."

        return generate_safe(prompt, 512)


class Evaluator:
    role = """SYSTEM INSTRUCTION: You are an objective Data Quality Auditor named Evaluator.
Verify whether the executed result satisfies the TARGET TASK.

Conclude your response strictly on a new line with:
VERDICT: SUCCESS | FINAL_ANSWER: [Extract exact answer here]
or
VERDICT: FAILED | REASON: [Short explanation of error]"""

    def generate(self, base_df, schema_context, step, answer_thinker, previous_code):
        performer = Performer()
        feedback_note = None
        observation = None
        answer_evaluator = "VERDICT: SUCCESS"
        clean_code = previous_code or ""
        redo = True
        rep_limit = 0

        while redo and rep_limit < 3:
            rep_limit += 1
            raw_code = performer.generate(
                schema_context,
                step,
                answer_thinker,
                feedback_note,
                previous_code,
            )

            clean_code = raw_code
            if "```" in clean_code:
                parts = clean_code.split("```")
                clean_code = parts[1]
            clean_code = re.sub(r"^(python|Python)\s*", "", clean_code).strip()

            lines = [
                l for l in clean_code.split("\n")
                if not l.strip().startswith(("import ", "from ", "df = pd.DataFrame"))
            ]
            clean_code = "\n".join(lines).strip()

            try:
                local_scope = {"df": base_df.copy(), "pd": pd, "np": np}
                exec(clean_code, globals(), local_scope)

                if "result_df" in local_scope:
                    observation = local_scope["result_df"]

                    if isinstance(observation, (pd.Series, pd.DataFrame)) and len(observation) == 1:
                        if isinstance(observation, pd.Series):
                            pass
                    redo = False
                else:
                    feedback_note = "Code ran but did not set 'result_df = ...'."
                    redo = True

            except Exception as e:
                feedback_note = f"Execution crashed with error: {str(e)}"
                redo = True

            if redo:
                audit_prompt = f"### ROLE:\n{self.role}\n\nTask: {step}\nFailing Code:\n{clean_code}\nError:\n{feedback_note}"
                answer_evaluator = generate_safe(audit_prompt, 256)

        successful_code = clean_code if not redo else previous_code
        return observation, answer_evaluator, successful_code

In [ ]:
thinker = Thinker()
evaluator = Evaluator()

In [ ]:
for i in [0, 130, 240, 360, 700]:
  df, col, data, answers, questions = load_ds(i)

  schema_context = f"Data Columns: {col}"

  temp1 = []
  temp2 = []

  match i:
    case 0:
      temp1 = [
        ["Identify the user with the highest net worth.","Identify if the user is self made."],
        ["Identify the youngest user.","Identify if the user is male."],
        ["Identify the city with the most users.","Identify if the city is in the United States."],
        ["Identify all users who belong to the 'Technology' category."],
        ["Identify the youngest user.","Identify the user's source of wealth."],
        ["Identify the four youngest users.","Identify the cities the users come from."]
      ]
      for l in [0,1,2,6,13,17]:
        temp2.append(answers[l])
    case 130:
      temp1 = [
        ["Identify if any users have a higher overall score than their potential score."],
        ["Identify all users who are under 18 years old."],
        ["Identify all players whose preferred foot is listed as their left foot.","Identify if any user's nationality starts with 'B'."],
        ["Identify the number of unique clubs."],
        ["Calculate the total value (in €) grouped by club.", "Identify the four clubs with the highest total value."],
        ["Calculate the average agility grouped by nationality.", "Identify the bottom 4 nationalities by average agility."]
      ]
      for l in [0,1,2,5,13,14]:
        temp2.append(answers[l])
    case 240:
      temp1 = [
        ["Identify all patients who do not experience exercise-induced angina."],
        ["Identify all patients who do not have normal resting electrocardiographic results."],
        ["Identify the maximum heart rate among all patients.","Compute the standard deviation of the maximum heart rate."],
        ["Identify the most common chest pain type by patient count."],
        ["Identify the 4 least common resting electrocardiographic results."],
        ["Identify the 5 oldest patients ordered from oldest to youngest."]
      ]
      for l in [0,3,7,8,13,16]:
        temp2.append(answers[l])
    case 360:
      temp1 = [
        ["Identify all satisfaction levels that are equal or lesser to 0.5."],
        ["Identify all employees who have been promoted in the last five years."],
        ["Identify the department with the highest count of employees."],
        ["Filter employees who had an accident at work","Identify the salary level with the smallest count of these employees."],
        ["Calculate the average satisfaction level by department.", "Identify the 3 departments with the lowest average satisfaction level."],
        ["Identify all employees who have been promoted in the last five years.","Identify the bottom 5 smallest monthly hours among employees."]
      ]
      for l in [2,6,8,11,14,18]:
        temp2.append(answers[l])
    case 700:
      temp1 = [
        ["Identify the song with the highest rank.","Identify if the song is from 1965."],
        ["Identify the song with the lowest rank.","Identify if the song has the word 'love' in its lyrics."],
        ["Identify all songs without lyrics."],
        ["Add a column counting occurrences of 'love' in lyrics.", "Identify the song with the highest count of 'love'."],
        ["Filter songs from the most recent year.", "Identify the top 4 songs by rank from that year."],
        ["Filter songs from the year 1965.", "Identify the song with the highest rank in 1965.", "Identify if that song is by the Beatles."]
      ]
      for l in [0,1,3,10,14,20]:
        temp2.append(answers[l])

  questions = temp1
  answers = temp2

  for q_idx in range(len(questions)):
        previous_code = ""
        step_result = None
        audit_summary = ""

        q_steps = questions[q_idx]
        final_result = None

        for step_num, s in enumerate(q_steps):
            answer_thinker = thinker.generate(s, schema_context)

            step_result, audit_summary, previous_code = evaluator.generate(
                df, schema_context, s, answer_thinker, previous_code
            )

        print(
        f"\n==================== NEW QUESTION: {q_steps} ====================")
        print(f"=== FINAL PIPELINE OUTPUT ===\n{step_result}\n")
        print(f"=== AUDITOR SUMMARY ===\n{audit_summary}\n")
        print(f"=== REAL BENCHMARK ANSWER ===\n{temp2[q_idx]}\n")
        print("-" * 74)

# **Unchecked Tri-Agent**

In [ ]:
def generate_safe(prompt: str, max_new_tokens: int = 512) -> str:
  messages = [{"role": "user", "content": prompt}]
  inputs = tokenizer.apply_chat_template(
      messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
  ).to(model.device)

  prompt_length = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

  return tokenizer.decode(
      outputs[0][prompt_length:], skip_special_tokens=True
  ).strip()

def load_ds(i):
    ds_id = all_qa["dataset"][i]
    df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/sample.parquet")

    df.columns = [re.sub(r'<gx:[^>]+>', '', str(c)).strip() for c in df.columns]
    col = list(df.columns)

    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype(str).str.slice(0, 100)

    df_str = df.to_csv(sep="|", index=False)

    qa_df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/qa.parquet")
    answers = qa_df["sample_answer"].tolist()
    questions = qa_df["question"].tolist()

    return df, col, df_str, answers, questions

In [ ]:
class Thinker:

  role = """You are a data scientist named Thinker. Read the Dataset Structure and explain concisely:
    1. Which column(s) to inspect.
    2. The exact value or aggregate operation required to complete the target task.
    Do NOT write code, another agent will write code based on the values you will give it. Keep explanations under 3 sentences."""

  def generate(self, step, schema_context):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"
    return generate_safe(prompt, 256)


class Performer:

  role = """You are a seasoned data scientist and python programmer named Performer.
      Your task is to use the Relevant Terms to produce pandas code that will realize the request of the TARGET TASK.
  CRITICAL CONSTRAINT RULES:
  1. Do NOT initialize, mock, copy, or create any new dataframes or dictionaries (e.g., NEVER write statements like `df = pd.DataFrame(...)`, `sample_df = ...`, or mock data arrays).
  2. If filtering for a subset or a single row, KEEP IT AS A DATAFRAME (e.g., use `df[df['col'] == val].head(1)` instead of `.iloc[0]` or `.loc[...]`) so subsequent chained steps can continue filtering it.
  3. You may use the original df or variables defined in the PREVIOUS WORKING CODE. Save the latest step's final output (DataFrame, Series, boolean, or scalar value) to result_df.
  4. Always add `na=False` inside your `.str.contains()` operations to avoid execution errors on empty data rows.
  5. Do NOT use pd.read_csv() or load any files.
  6. Wrap your code inside ```python and ``` blocks."""

  def generate(
      self, schema_context, step, answer_thinker, previous_code
  ):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### STRATEGIC TASK:\n{step}\n\n### RELEVANT TERMS:\n{answer_thinker}"
    if previous_code:
      prompt += f"\n\n### PREVIOUS WORKING CODE (Build on top of this):\n```python\n{previous_code}\n```"
    return generate_safe(prompt, 512)


def execute_code_direct(raw_code, base_df):
  clean_code = raw_code
  if "```" in clean_code:
    parts = clean_code.split("```")
    clean_code = parts[1]
  elif "---" in clean_code:
    parts = clean_code.split("---")
    clean_code = parts[1] if len(parts) > 1 else parts[0]

  clean_code = re.sub(r"^(python|Python)\s*", "", clean_code).strip()

  lines = [
      l
      for l in clean_code.split("\n")
      if not l.strip().startswith(("import ", "from ", "df = pd.DataFrame", "df="))
  ]
  clean_code = "\n".join(lines).strip()

  local_scope = {"df": base_df.copy(), "pd": pd, "np": np}
  try:
    exec(clean_code, globals(), local_scope)

    target_val = None
    if "result_df" in local_scope:
      target_val = local_scope["result_df"]
    else:
      assigned = [
          k
          for k in local_scope
          if k not in ["df", "pd", "np", "__builtins__"]
          and not callable(local_scope[k])
      ]
      if assigned:
        target_val = local_scope[assigned[-1]]

    if target_val is not None:
      if isinstance(target_val, pd.Series) and len(target_val) == 1:
        target_val = target_val.iloc[0]
      return target_val, clean_code

    return "Executed (No result variable found)", clean_code
  except Exception as e:
    return f"Execution Error: {str(e)}", clean_code

In [ ]:
thinker = Thinker()
performer = Performer()

In [ ]:
for i in [0, 130, 240, 360, 700]:
  df, col, data, answers, questions = load_ds(i)

  schema_context = f"Data Columns: {col}"

  temp1 = []
  temp2 = []

  match i:
    case 0:
      temp1 = [
        ["Identify the user with the highest net worth.","Identify if the user is self made."],
        ["Identify the youngest user.","Identify if the user is male."],
        ["Identify the city with the most users.","Identify if the city is in the United States."],
        ["Identify all users who belong to the 'Technology' category."],
        ["Identify the youngest user.","Identify the user's source of wealth."],
        ["Identify the four youngest users.","Identify the cities the users come from."]
      ]
      for l in [0,1,2,6,13,17]:
        temp2.append(answers[l])
    case 130:
      temp1 = [
        ["Identify if any users have a higher overall score than their potential score."],
        ["Identify all users who are under 18 years old."],
        ["Identify all players whose preferred foot is listed as their left foot.","Identify if any user's nationality starts with 'B'."],
        ["Identify the number of unique clubs."],
        ["Calculate the total value (in €) grouped by club.", "Identify the four clubs with the highest total value."],
        ["Calculate the average agility grouped by nationality.", "Identify the bottom 4 nationalities by average agility."]
      ]
      for l in [0,1,2,5,13,14]:
        temp2.append(answers[l])
    case 240:
      temp1 = [
        ["Identify all patients who do not experience exercise-induced angina."],
        ["Identify all patients who do not have normal resting electrocardiographic results."],
        ["Identify the maximum heart rate among all patients.","Compute the standard deviation of the maximum heart rate."],
        ["Identify the most common chest pain type by patient count."],
        ["Identify the 4 least common resting electrocardiographic results."],
        ["Identify the 5 oldest patients ordered from oldest to youngest."]
      ]
      for l in [0,3,7,8,13,16]:
        temp2.append(answers[l])
    case 360:
      temp1 = [
        ["Identify all satisfaction levels that are equal or lesser to 0.5."],
        ["Identify all employees who have been promoted in the last five years."],
        ["Identify the department with the highest count of employees."],
        ["Filter employees who had an accident at work","Identify the salary level with the smallest count of these employees."],
        ["Calculate the average satisfaction level by department.", "Identify the 3 departments with the lowest average satisfaction level."],
        ["Identify all employees who have been promoted in the last five years.","Identify the bottom 5 smallest monthly hours among employees."]
      ]
      for l in [2,6,8,11,14,18]:
        temp2.append(answers[l])
    case 700:
      temp1 = [
        ["Identify the song with the highest rank.","Identify if the song is from 1965."],
        ["Identify the song with the lowest rank.","Identify if the song has the word 'love' in its lyrics."],
        ["Identify all songs without lyrics."],
        ["Add a column counting occurrences of 'love' in lyrics.", "Identify the song with the highest count of 'love'."],
        ["Filter songs from the most recent year.", "Identify the top 4 songs by rank from that year."],
        ["Filter songs from the year 1965.", "Identify the song with the highest rank in 1965.", "Identify if that song is by the Beatles."]
      ]
      for l in [0,1,3,10,14,20]:
        temp2.append(answers[l])

  questions = temp1
  answers = temp2

  for q_idx in range(len(questions)):
        q_steps = questions[q_idx]
        previous_code = ""
        final_result = None

        for step_num, s in enumerate(q_steps):
            answer_thinker = thinker.generate(s, schema_context)

            answer_performer = performer.generate(schema_context, s, answer_thinker, previous_code)

            final_result, previous_code = execute_code_direct(answer_performer, df)

        print(f"\n==================== NEW QUESTION: {q_steps} ====================")
        print(f"=== FINAL EXECUTED RESULT ===\n{final_result}\n")
        print(f"=== REAL BENCHMARK ANSWER ===\n{answers[q_idx]}\n")
        print("-" * 74)

# **Auto-Checked Tri-Agent**

In [ ]:
def generate_safe(prompt: str, max_new_tokens: int = 512) -> str:
  messages = [{"role": "user", "content": prompt}]
  inputs = tokenizer.apply_chat_template(
      messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
  ).to(model.device)

  prompt_length = inputs["input_ids"].shape[1]

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

  return tokenizer.decode(
      outputs[0][prompt_length:], skip_special_tokens=True
  ).strip()


def load_ds(i):
    ds_id = all_qa["dataset"][i]
    df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/sample.parquet")

    df.columns = [re.sub(r'<gx:[^>]+>', '', str(c)).strip() for c in df.columns]
    col = list(df.columns)

    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype(str).str.slice(0, 100)

    df_str = df.to_csv(sep="|", index=False)

    qa_df = pd.read_parquet(f"hf://datasets/cardiffnlp/databench/data/{ds_id}/qa.parquet")
    answers = qa_df["sample_answer"].tolist()
    questions = qa_df["question"].tolist()

    return df, col, df_str, answers, questions

In [ ]:
class Thinker:

  role = """You are a data scientist named Thinker. Read the Dataset Structure and explain concisely:
1. Which column(s) to inspect.
2. The exact value or aggregate operation required to complete the target task.
Do NOT write code. Keep explanations under 3 sentences."""

  def generate(self, step, schema_context):
    prompt = f"### ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"
    return generate_safe(prompt, 256)


class Performer:

  role = """You are a seasoned data scientist and python programmer named Performer.
      Your task is to use the Relevant Terms to produce pandas code that will realize the request of the TARGET TASK.
  CRITICAL CONSTRAINT RULES:
  1. Do NOT initialize, mock, copy, or create any new dataframes or dictionaries (e.g., NEVER write statements like `df = pd.DataFrame(...)`, `sample_df = ...`, or mock data arrays).
  2. If filtering for a subset or a single row, KEEP IT AS A DATAFRAME (e.g., use `df[df['col'] == val].head(1)` instead of `.iloc[0]` or `.loc[...]`) so subsequent chained steps can continue filtering it.
  3. You may use the original df or variables defined in the PREVIOUS WORKING CODE. Save the latest step's final output (DataFrame, Series, boolean, or scalar value) to result_df.
  4. Always add `na=False` inside your `.str.contains()` operations to avoid execution errors on empty data rows.
  5. Do NOT use pd.read_csv() or load any files.
  6. Wrap your code inside ```python and ``` blocks."""

  def generate(
      self, schema_context, step, answer_thinker, feedback_note, previous_code
  ):
    prompt = f"### ROLE:\n{self.role}\n\n### SCHEMA:\n{schema_context}\n\n### TARGET TASK:\n{step}\n\n### METHOD:\n{answer_thinker}"

    if previous_code:
      prompt += f"\n\n### PREVIOUS WORKING CODE:\n```python\n{previous_code}\n```\nBuild directly on top of variables from the previous code."

    if feedback_note:
      prompt += f"\n\n### RUNTIME ERROR FEEDBACK:\n{feedback_note}\nFix the code based on this error."

    return generate_safe(prompt, 512)


class AutoDebugger:
  """Executes generated code against the runtime environment.

  Uses Python execution feedback for retries without invoking an Evaluator
  model.
  """

  def generate(
      self, base_df, schema_context, step, answer_thinker, previous_code
  ):
    performer = Performer()
    feedback_note = None
    observation = None
    clean_code = previous_code or ""
    redo = True
    rep_limit = 0

    while redo and rep_limit < 3:
      rep_limit += 1
      raw_code = performer.generate(
          schema_context,
          step,
          answer_thinker,
          feedback_note,
          previous_code,
      )

      clean_code = raw_code
      if "```" in clean_code:
        parts = clean_code.split("```")
        clean_code = parts[1]
      clean_code = re.sub(r"^(python|Python)\s*", "", clean_code).strip()

      lines = [
          l
          for l in clean_code.split("\n")
          if not l.strip().startswith(("import ", "from ", "df = pd.DataFrame"))
      ]
      clean_code = "\n".join(lines).strip()

      try:
        local_scope = {"df": base_df.copy(), "pd": pd, "np": np}
        exec(clean_code, globals(), local_scope)

        if "result_df" in local_scope:
          observation = local_scope["result_df"]

          if (
              isinstance(observation, (pd.Series, pd.DataFrame))
              and len(observation) == 1
          ):
            if isinstance(observation, pd.Series):
              observation = observation.iloc[0]
          redo = False
        else:
          feedback_note = "Code ran but did not set 'result_df = ...'."
          redo = True

      except Exception as e:
        feedback_note = f"Execution crashed with error: {str(e)}"
        redo = True

    successful_code = clean_code if not redo else previous_code
    return observation, successful_code

In [ ]:
thinker = Thinker()
executor = AutoDebugger()

In [ ]:
for i in [0, 130, 240, 360, 700]:
  df, col, data, answers, questions = load_ds(i)

  schema_context = f"Data Columns: {col}"

  temp1 = []
  temp2 = []

  match i:
    case 0:
      temp1 = [
        ["Identify the user with the highest net worth.","Identify if the user is self made."],
        ["Identify the youngest user.","Identify if the user is male."],
        ["Identify the city with the most users.","Identify if the city is in the United States."],
        ["Identify all users who belong to the 'Technology' category."],
        ["Identify the youngest user.","Identify the user's source of wealth."],
        ["Identify the four youngest users.","Identify the cities the users come from."]
      ]
      for l in [0,1,2,6,13,17]:
        temp2.append(answers[l])
    case 130:
      temp1 = [
        ["Identify if any users have a higher overall score than their potential score."],
        ["Identify all users who are under 18 years old."],
        ["Identify all players whose preferred foot is listed as their left foot.","Identify if any user's nationality starts with 'B'."],
        ["Identify the number of unique clubs."],
        ["Calculate the total value (in €) grouped by club.", "Identify the four clubs with the highest total value."],
        ["Calculate the average agility grouped by nationality.", "Identify the bottom 4 nationalities by average agility."]
      ]
      for l in [0,1,2,5,13,14]:
        temp2.append(answers[l])
    case 240:
      temp1 = [
        ["Identify all patients who do not experience exercise-induced angina."],
        ["Identify all patients who do not have normal resting electrocardiographic results."],
        ["Identify the maximum heart rate among all patients.","Compute the standard deviation of the maximum heart rate."],
        ["Identify the most common chest pain type by patient count."],
        ["Identify the 4 least common resting electrocardiographic results."],
        ["Identify the 5 oldest patients ordered from oldest to youngest."]
      ]
      for l in [0,3,7,8,13,16]:
        temp2.append(answers[l])
    case 360:
      temp1 = [
        ["Identify all satisfaction levels that are equal or lesser to 0.5."],
        ["Identify all employees who have been promoted in the last five years."],
        ["Identify the department with the highest count of employees."],
        ["Filter employees who had an accident at work","Identify the salary level with the smallest count of these employees."],
        ["Calculate the average satisfaction level by department.", "Identify the 3 departments with the lowest average satisfaction level."],
        ["Identify all employees who have been promoted in the last five years.","Identify the bottom 5 smallest monthly hours among employees."]
      ]
      for l in [2,6,8,11,14,18]:
        temp2.append(answers[l])
    case 700:
      temp1 = [
        ["Identify the song with the highest rank.","Identify if the song is from 1965."],
        ["Identify the song with the lowest rank.","Identify if the song has the word 'love' in its lyrics."],
        ["Identify all songs without lyrics."],
        ["Add a column counting occurrences of 'love' in lyrics.", "Identify the song with the highest count of 'love'."],
        ["Filter songs from the most recent year.", "Identify the top 4 songs by rank from that year."],
        ["Filter songs from the year 1965.", "Identify the song with the highest rank in 1965.", "Identify if that song is by the Beatles."]
      ]
      for l in [0,1,3,10,14,20]:
        temp2.append(answers[l])

  questions = temp1
  answers = temp2

  for q_idx in range(len(questions)):
    previous_code = ""
    step_result = None

    q_steps = questions[q_idx]

    for step_num, s in enumerate(q_steps):
      answer_thinker = thinker.generate(s, schema_context)

      step_result, previous_code = executor.generate(
          df, schema_context, s, answer_thinker, previous_code
      )

    print(f"\n==================== NEW QUESTION: {q_steps} ====================")
    print(f"=== FINAL PIPELINE OUTPUT ===\n{step_result}\n")
    print(f"=== REAL BENCHMARK ANSWER ===\n{temp2[q_idx]}\n")
    print("-" * 74)